In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
from matplotlib.colors import LogNorm
from pathlib import Path
import imageio.v3 as iio  # TIFF reader
import tracemalloc

In [ ]:
def qx_qy_map(shape, cx_pix, cy_pix, pix_um, L_mm, lam_A):
    nrows, ncols = shape
    k = 2.0 * np.pi / lam_A
    px_mm = pix_um * 1e-3
    yy, xx = np.indices((nrows, ncols), dtype=np.float64)
    x_mm = (xx - cx_pix) * px_mm
    y_mm = (yy - cy_pix) * px_mm
    r = np.sqrt(x_mm**2 + y_mm**2 + L_mm**2)
    qx = k * x_mm / r
    qy = k * y_mm / r
    return qx, qy

def read_tif_raw(path: Path) -> np.ndarray:
    arr = iio.imread(path)
    arr = np.asarray(arr)
    if arr.ndim == 3:
        if arr.shape[-1] in (3, 4):
            arr = arr[..., :3].mean(axis=-1)
        else:
            arr = arr[0]
    return arr.astype(np.float32, copy=False)

def save_qxqy_image(img, qx, qy, out_png, show=False, q_range_x=None, q_range_y=None, labels=True):
    qy = -qy  # Flip so positive is upward

    qx_edges = 0.5 * (qx[:-1, :-1] + qx[:-1, 1:])
    qy_edges = 0.5 * (qy[:-1, :-1] + qy[1:, :-1])

    finite = img[np.isfinite(img) & (img > 0)]
    if finite.size > 0:
        vmin = float(np.percentile(finite, 1))
        vmax = float(np.percentile(finite, 99))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin <= 0 or vmin >= vmax:
            vmin, vmax = 1e-6, float(np.nanmax(finite)) if np.isfinite(np.nanmax(finite)) else 1.0
            if vmin >= vmax:
                vmax = vmin * 10.0
    else:
        vmin, vmax = 1e-6, 1.0

    img_clipped = np.clip(img, vmin, vmax)

    qx_range = np.nanmax(qx) - np.nanmin(qx)
    qy_range = np.nanmax(qy) - np.nanmin(qy)
    base_height = 5
    aspect_ratio = qx_range / qy_range if qy_range != 0 else 1
    figsize = (base_height * aspect_ratio, base_height)

    # Use the object-oriented API
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect('equal', adjustable='box')

    # Apply q-range limits if provided
    if q_range_x is not None:
        plt.xlim(q_range_x)
    if q_range_y is not None:
        plt.ylim(q_range_y)

    im = ax.pcolormesh(qx_edges, qy_edges, img_clipped[:-1, :-1],
                       norm=LogNorm(vmin=vmin, vmax=vmax),
                       shading='auto')

    if labels:
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Log(Intensity)")
        ax.set_xlabel("qx ($\mathrm{{\AA}}^{{-1}}$)")
        ax.set_ylabel("qy ($\mathrm{{\AA}}^{{-1}}$)")
        ax.set_title("2D qx–qy Map")
    else:
        ax.axis("off")

    fig.savefig(out_png, dpi=150, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    del fig, ax, im
    gc.collect()

def process_tif_folder(input_folder, result_folder, cx_pix, cy_pix, sample_idx=0,
                       q_range_x=None, q_range_y=None, labels=True):
    folder = Path(input_folder)
    files = sorted([f for f in folder.iterdir() if f.suffix.lower() in (".tif", ".tiff")])

    if not files:
        print("No TIFF files found in the folder (searched for .tif and .tiff).")
        return
    for i, f in enumerate(files):
        # Track current and peak memory usage every 10 iterations to monitor leaks
        if i % 10 == 0:
            current, peak = tracemalloc.get_traced_memory()
            print(f"[{i}] Current memory: {current / 1e6:.1f} MB; Peak: {peak / 1e6:.1f} MB")
        # Force garbage collection every 20 iterations to release memory from Matplotlib/colorbar leaks 
        if i % 20 == 0:
            gc.collect()
        img = read_tif_raw(f)
        print(f"Processing image {f.stem}")
        qx, qy = qx_qy_map(img.shape, cx_pix=cx_pix, cy_pix=cy_pix,
                           pix_um=pix_um, L_mm=L_mm, lam_A=lam_A)
        out_png = Path(result_folder) / f"{f.stem}_qxqy.png"
        show_flag = (i == sample_idx)
        save_qxqy_image(img, qx, qy, out_png, show=show_flag,
                        q_range_x=q_range_x, q_range_y=q_range_y, labels=labels)

In [ ]:
input_folder = "../blends_tiff/"    # Folder containing TIFF files
result_folder = "./outputs/"  # Folder for output PNG images
sample_idx = 0             # Index of file to display (0 = first)

# Detector geometry (beam center REQUIRED)
L_mm   = 5030
lam_A  = 0.9184
pix_um = 172.0
cx_pix = 754
cy_pix = 1082

# Optional customizations
q_range_x = (-0.06, 0.06)   
q_range_y = (-0.04, 0.1)   
show_labels = False        

# Create results folder
Path(result_folder).mkdir(exist_ok=True)

tracemalloc.start()

process_tif_folder(input_folder, result_folder, cx_pix, cy_pix, sample_idx,
                   q_range_x=q_range_x, q_range_y=q_range_y, labels=show_labels)